## Load EC3 EPD Data

### Load dataframe

In [25]:
import pandas as pd

# Load in epd_data_all.csv to a dataframe
df = pd.read_csv('../02_processed_data/epd_data_all.csv')


In [26]:
df.head()

,gwp_per_category_declared_unit,name,created_on,standard_deviation,updated_on,id,open_xpd_uuid,gwp,lightweight,uncertainty_factor,...,plant_or_group.latitude,plant_or_group.type,concrete_compressive_strength_other,concrete_compressive_strength_other_d,cementitious.fly_ash,concrete_aggregate_size_max,cementitious.ggbs,gwp_val_per_cy,created_date_formatted,Compressive_Strength
0,138.14156 kgCO2e,8004FF,2026-01-22T18:01:50.790474Z,15.52357137 kgCO2e,2026-01-22T18:01:52.526114Z,2a6cb293de444f09ae27b0ad5671875e,wapw6cep,138.14156 kgCO2e,False,1.094574,...,39.620020,Plant,NaN,NaN,NaN,NaN,NaN,105.585046,2026-01-22 18:01:50.790474+00:00,0
1,169 kgCO2e,Mix 4003592,2026-01-20T22:49:54.486975Z,20.14151047 kgCO2e,2026-01-21T18:19:27.512744Z,de0f439571f046a3a25c74990065f9d5,ec3dq2yg,169 kgCO2e,NaN,1.100302,...,32.422152,Plant,NaN,NaN,NaN,NaN,NaN,129.209795,2026-01-20 22:49:54.486975+00:00,0
2,96.92 kgCO2e,CDFPUMP,2026-01-20T22:55:58.190134Z,11.17904035 kgCO2e,2026-01-21T17:37:28.774255Z,88d184e66333401aad262f3cb9bd784a,ec384n78,96.92 kgCO2e,NaN,1.097073,...,39.988902,Plant,NaN,NaN,NaN,NaN,NaN,74.085380,2026-01-20 22:55:58.190134+00:00,0
3,124 kgCO2e,Mix CDFTYPE2,2026-01-20T22:59:24.372401Z,14.77838638 kgCO2e,2026-01-21T17:04:24.342601Z,d6eff2b999b749cdae72b564436bfb56,ec3bm89s,124 kgCO2e,NaN,1.100302,...,40.077776,Plant,NaN,NaN,NaN,NaN,NaN,94.804820,2026-01-20 22:59:24.372401+00:00,0
4,135.11678 kgCO2e,2 Sack Slurry,2026-01-05T15:27:39.641857Z,15.18366361 kgCO2e,2026-01-05T15:27:41.124971Z,7a08020714c2490a9213e546b111077c,wapaprnp,135.11678 kgCO2e,False,1.094574,...,31.379164,Plant,NaN,NaN,NaN,NaN,NaN,103.291381,2026-01-05 15:27:39.641857+00:00,0


In [27]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Filter to 2500–8000 psi inclusive; drop anything outside that range
df_filtered = df[
    (df['Compressive_Strength'] >= 2500) & (df['Compressive_Strength'] <= 8000)
].copy()

# Known SCM columns; any other cementitious.* columns in the dataframe are "other" SCMs
KNOWN_SCM_COLS = {'cementitious.fly_ash', 'cementitious.ggbs'}
other_scm_cols = [c for c in df_filtered.columns
                  if c.startswith('cementitious.') and c not in KNOWN_SCM_COLS]

def classify_scm(row):
    has_fly_ash = pd.notna(row.get('cementitious.fly_ash')) and row.get('cementitious.fly_ash') > 0
    has_ggbs    = pd.notna(row.get('cementitious.ggbs'))    and row.get('cementitious.ggbs')    > 0
    has_other   = any(
        pd.notna(row.get(c)) and row.get(c) > 0
        for c in other_scm_cols
    )

    if has_fly_ash and has_ggbs:
        return 'Both Fly Ash & Slag'
    elif has_fly_ash:
        return 'Fly Ash'
    elif has_ggbs:
        return 'Slag'
    elif has_other:
        return 'Other'   # non-fly-ash / non-slag SCM present; not plotted
    else:
        return 'No SCM'  # no cementitious supplementary material detected

df_filtered['scm_type'] = df_filtered.apply(classify_scm, axis=1)
df_filtered['gwp_rounded'] = df_filtered['gwp_val_per_cy'].round(0)

# Categories to PLOT (Other is excluded from the chart but retained in data)
scm_plot_order = ['No SCM', 'Fly Ash', 'Slag', 'Both Fly Ash & Slag']
colors         = ['#2c7fb8', '#41b6c4', '#7fcdbb', '#c7e9b4']
color_map      = dict(zip(scm_plot_order, colors))

df_plot = df_filtered[df_filtered['scm_type'].isin(scm_plot_order)].copy()

# Map each strength bucket to a numeric index so we can control box positions exactly
strength_order  = sorted(df_plot['Compressive_Strength'].unique())
strength_to_idx = {s: i for i, s in enumerate(strength_order)}

# Evenly distribute 4 box groups within each category (width 0.8)
n_groups    = len(scm_plot_order)
group_width = 0.8
box_width   = group_width / n_groups
offsets_map = {
    scm: (i - (n_groups - 1) / 2) * box_width
    for i, scm in enumerate(scm_plot_order)
}  # roughly -0.3, -0.1, +0.1, +0.3

# Jitter and dot settings
POINT_JITTER_WIDTH  = 0.04   # uniform random x-offset applied to all scatter dots
MAX_DOTS_PER_BUCKET = 600    # cap for No SCM only; other buckets show all points

fig = go.Figure()
rng = np.random.default_rng(seed=42)

# --- Box traces (no built-in points; dots drawn separately for consistent jitter) ---
for scm_type, color in zip(scm_plot_order, colors):
    subset   = df_plot[df_plot['scm_type'] == scm_type]
    x_values = [strength_to_idx[s] + offsets_map[scm_type]
                for s in subset['Compressive_Strength']]

    fig.add_trace(go.Box(
        x=x_values,
        y=subset['gwp_rounded'],
        name=scm_type,
        marker_color=color,
        line_color=color,
        fillcolor=color,
        boxpoints=False,   # dots drawn as scatter traces below for uniform jitter
        width=box_width * 0.6,
        marker=dict(size=5, opacity=0.6, line=dict(width=0.5, color='white')),
        customdata=subset[['name']].values,
        hovertemplate='<b>%{customdata[0]}</b><br>GWP: %{y} kg CO₂e/cy<extra>' + scm_type + '</extra>',
    ))

# --- Scatter dot traces (one per SCM type, with uniform POINT_JITTER_WIDTH) ---
for scm_type, color in zip(scm_plot_order, colors):
    subset    = df_plot[df_plot['scm_type'] == scm_type]
    is_no_scm = (scm_type == 'No SCM')
    sampled_x, sampled_y, sampled_names = [], [], []

    for strength in strength_order:
        bucket = subset[subset['Compressive_Strength'] == strength]
        n = min(MAX_DOTS_PER_BUCKET, len(bucket)) if is_no_scm else len(bucket)
        if n == 0:
            continue
        sample = bucket.sample(n=n, random_state=42) if is_no_scm else bucket
        x_center = strength_to_idx[strength] + offsets_map[scm_type]
        jitter = rng.uniform(-POINT_JITTER_WIDTH, POINT_JITTER_WIDTH, n)
        sampled_x.extend(x_center + jitter)
        sampled_y.extend(sample['gwp_rounded'].values)
        sampled_names.extend(sample['name'].values)

    opacity = 0.35 if is_no_scm else 0.6
    fig.add_trace(go.Scatter(
        x=sampled_x,
        y=sampled_y,
        mode='markers',
        marker=dict(color=color, size=5, opacity=opacity,
                    line=dict(width=0.5, color='white')),
        customdata=[[n] for n in sampled_names],
        hovertemplate='<b>%{customdata[0]}</b><br>GWP: %{y} kg CO₂e/cy<extra>' + scm_type + '</extra>',
        showlegend=False,
        name=scm_type + ' (dots)',
    ))

# --- Count annotations at top of each whisker (layout annotations support textangle) ---
for scm_type in scm_plot_order:
    for strength in strength_order:
        subset = df_plot[
            (df_plot['scm_type'] == scm_type) &
            (df_plot['Compressive_Strength'] == strength)
        ]
        if len(subset) == 0:
            continue

        count = len(subset)
        vals = subset['gwp_rounded']
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        whisker_top = min(q3 + 1.5 * (q3 - q1), vals.max())

        fig.add_annotation(
            x=strength_to_idx[strength] + offsets_map[scm_type],
            y=whisker_top,
            text=str(count),
            textangle=-90,
            showarrow=False,
            yanchor='bottom',
            font=dict(size=9, color='#444444'),
            xref='x', yref='y',
        )

# --- Footnotes ---
fig.add_annotation(
    text=(
        'Numbers at tops of whiskers denote count of concrete EPDs<br>'
        'Random sampling of dots for No SCM mixes removed for visual clarity'
    ),
    xref='paper', yref='paper',
    x=1.0, y=-0.13,
    xanchor='right', yanchor='top',
    showarrow=False,
    font=dict(size=10, color='#888888'),
    align='right',
)

# --- Vertical separators between strength buckets ---
y_min = df_plot['gwp_rounded'].min()
y_max = df_plot['gwp_rounded'].max()

for i in range(len(strength_order) - 1):
    fig.add_shape(
        type='line',
        x0=i + 0.5, y0=y_min - 20,
        x1=i + 0.5, y1=y_max + 60,
        line=dict(color='rgba(128,128,128,0.3)', width=1, dash='dash'),
        layer='below'
    )

# --- Highlight box: Fly Ash 6,000 psi ---
fa_6000_vals = df_plot[
    (df_plot['scm_type'] == 'Fly Ash') & (df_plot['Compressive_Strength'] == 6000)
]['gwp_rounded']
fa_6000_q1 = fa_6000_vals.quantile(0.25)
fa_6000_q3 = fa_6000_vals.quantile(0.75)
fa_6000_whisker_top = min(fa_6000_q3 + 1.5 * (fa_6000_q3 - fa_6000_q1), fa_6000_vals.max())

x_center_fa_6000 = strength_to_idx[6000] + offsets_map['Fly Ash']
box_half  = box_width * 0.6 / 2  # half-width of the box trace (0.06)
pad_x     = 0.025                 # keeps ~0.055 gap to adjacent bucket boxes
pad_y_bot = 5
pad_y_top = 30                    # extra headroom above whisker_top to enclose count annotation

fig.add_shape(
    type='rect',
    x0=x_center_fa_6000 - box_half - pad_x,
    x1=x_center_fa_6000 + box_half + pad_x,
    y0=fa_6000_vals.min() - pad_y_bot,
    y1=fa_6000_whisker_top + pad_y_top,
    line=dict(color='#E75A7C', width=2, dash='dot'),
    fillcolor='rgba(0,0,0,0)',
    layer='above',
)

# --- Axes ---
fig.update_xaxes(
    tickvals=list(range(len(strength_order))),
    ticktext=[f'{int(s)} psi' for s in strength_order],
    tickangle=-45,
    title_text='Compressive Strength',
    range=[-0.5, len(strength_order) - 0.5],
)

fig.update_yaxes(
    title_text='GWP (kg CO₂e per cubic yard)',
    showgrid=True,
    gridcolor='lightgrey',
    range=[y_min - 20, y_max + 80],
)

fig.update_layout(
    title={
        'text': 'GWP Distribution by Compressive Strength and SCM Type',
        'x': 0.5, 'xanchor': 'center', 'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'},
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    width=1400,
    margin=dict(b=120),
    legend={
        'title': 'SCM Type',
        'orientation': 'v',
        'yanchor': 'top', 'y': 0.99,
        'xanchor': 'left', 'x': 1.01,
    },
)

# --- Summary ---
print(f"Total samples (2500–8000 psi): {len(df_filtered)}")
print(f"  Plotted (excl. Other): {len(df_plot)}")
if other_scm_cols:
    print(f"  Other SCM columns detected: {other_scm_cols}")
else:
    print(f"  No additional cementitious.* columns found — 'Other' bucket will always be empty with current data")
print(f"\nSamples by SCM Type (all, incl. Other):")
print(df_filtered['scm_type'].value_counts())
print(f"\nNo SCM dots shown per strength bucket (capped at {MAX_DOTS_PER_BUCKET}):")
no_scm_subset = df_plot[df_plot['scm_type'] == 'No SCM']
for strength in strength_order:
    total = len(no_scm_subset[no_scm_subset['Compressive_Strength'] == strength])
    shown = min(MAX_DOTS_PER_BUCKET, total)
    print(f"  {int(strength)} psi: {shown} shown / {total} total")

fig.show()


Total samples (2500–8000 psi): 72169
  Plotted (excl. Other): 72169
  No additional cementitious.* columns found — 'Other' bucket will always be empty with current data

Samples by SCM Type (all, incl. Other):
scm_type
No SCM                 67272
Fly Ash                 3932
Slag                     874
Both Fly Ash & Slag       91
Name: count, dtype: int64

No SCM dots shown per strength bucket (capped at 600):
  2500 psi: 600 shown / 1619 total
  3000 psi: 600 shown / 10810 total
  3500 psi: 600 shown / 7198 total
  4000 psi: 600 shown / 19012 total
  4500 psi: 600 shown / 8211 total
  5000 psi: 600 shown / 12213 total
  5500 psi: 540 shown / 540 total
  6000 psi: 600 shown / 4520 total
  6500 psi: 490 shown / 490 total
  7000 psi: 600 shown / 1072 total
  7500 psi: 386 shown / 386 total
  8000 psi: 600 shown / 1201 total


In [28]:
from plotly.subplots import make_subplots

# Build a row of pie charts — one per strength bucket — showing SCM type breakdown.
# Colors and category order match the main chart above; no additional legend needed.

fig_pie = make_subplots(
    rows=1,
    cols=len(strength_order),
    specs=[[{'type': 'pie'}] * len(strength_order)],
    subplot_titles=[f'{int(s)} psi' for s in strength_order],
    horizontal_spacing=0.005,
)

for col_idx, strength in enumerate(strength_order, start=1):
    bucket = df_plot[df_plot['Compressive_Strength'] == strength]
    counts = bucket['scm_type'].value_counts()

    labels, values, pie_colors = [], [], []
    for scm_type, color in zip(scm_plot_order, colors):
        count = counts.get(scm_type, 0)
        if count > 0:
            labels.append(scm_type)
            values.append(count)
            pie_colors.append(color)

    total = sum(values)
    pct_texts = [f'{round(v / total * 100)}%' for v in values]

    fig_pie.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            marker=dict(colors=pie_colors),
            text=pct_texts,
            textinfo='text',
            hovertemplate='%{label}<br>%{text} of %{value:,} mixes<extra></extra>',
            showlegend=False,
            textfont=dict(size=10),
            insidetextorientation='auto',
        ),
        row=1, col=col_idx,
    )

fig_pie.update_layout(
    title={
        'text': 'Percentages of EPDs by SCM Type within Each Strength Bucket',
        'x': 0.5, 'xanchor': 'center', 'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'},
    },
    height=300,
    width=1400,
    paper_bgcolor='white',
    margin=dict(t=110, b=20, l=10, r=10),
)


fig_pie.show()


### Save Plots

In [29]:
# Save the plot as HTML
import os
output_path = '../05_outputs/gwp_by_compressive_strength_scm_v2.html'
fig.write_html(output_path, include_plotlyjs='cdn')

# Save the pie chart
output_pie_path = '../05_outputs/scm_type_breakdown_pies.html'
fig_pie.write_html(output_pie_path, include_plotlyjs='cdn')

In [30]:
# Save high-resolution PNG versions to the tests folder
# Use Kaleido directly with n=4 to handle Chrome opening multiple tabs on startup,
# and await it directly to avoid the sync-wrapper conflict in Jupyter's event loop.
from kaleido import Kaleido

async with Kaleido(n=4) as k:
    await k.write_fig(fig, path='../tests/gwp_by_compressive_strength_scm.png',
                      opts={'format': 'png', 'scale': 3, 'width': fig.layout.width, 'height': fig.layout.height})
    await k.write_fig(fig_pie, path='../tests/scm_type_breakdown_pies.png',
                      opts={'format': 'png', 'scale': 3, 'width': fig_pie.layout.width, 'height': fig_pie.layout.height})
print("Saved PNGs to tests/")

Saved PNGs to tests/
